<a href="https://colab.research.google.com/github/AKDGrant/automation-news-bot/blob/main/NewsDigest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily News Summarizer Bot
Pulls top headlines from Hacker News and summarizes them automatically using a free AI model.
This generates a daily digest that's ready for quick reading.


In [ ]:
import requests
!pip install feedparser
!pip install transformers
import feedparser
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [2]:
# ======== RSS Feed ========
# we're using Hacker News RSS since it's free and always has fresh tech news
rss_url = "https://news.ycombinator.com/rss"

In [3]:
# ======== Function to grab top headlines ========
def fetch_headlines(rss_url, max_items=5):
    """
    Go to the RSS feed and pull out the top headlines.
    We'll just take a few so it's manageable.
    """
    feed = feedparser.parse(rss_url)
    headlines = [entry.title for entry in feed.entries[:max_items]]
    return headlines

In [ ]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

def summarize(text):
    """
    take a headline and make it short and easy to read.
    basically, we're letting AI do the hard work of summarizing
    """
    output = pipe(text, max_new_tokens=100, do_sample=False)
    return output[0]['generated_text']

In [5]:
def daily_news_report(rss_url, max_items=5, filename="daily_news_report.txt"):
    """
    this is the main function:
    1. grab top headlines
    2. summarize them
    3. save everything to a text file so you can screenshot or demo it
    """
    headlines = fetch_headlines(rss_url, max_items)
    report_lines = []

    for idx, headline in enumerate(headlines, 1):
        summary = summarize(headline)
        report_lines.append(f"{idx}. Original: {headline}\n   Summary: {summary}\n")

    report_text = "\n".join(report_lines)

    with open(filename, "w") as f:
        f.write(report_text)

    print(f"Daily news report saved to {filename}")
    print(report_text)

# ======== Run it once to see it in action ========
daily_news_report(rss_url, max_items=5)

Daily news report saved to daily_news_report.txt
1. Original: SimpleFold: Folding proteins is simpler than you think
   Summary: SimpleFold: Folding proteins is simpler than you think.

2. Original: New math revives geometry's oldest problems
   Summary: New math revives geometry's oldest problems.

3. Original: Open Social
   Summary: Open Social

4. Original: The Beauty of Programming (2001)
   Summary: The Beauty of Programming (2001)

5. Original: Why do we remember some life moments but not others?
   Summary: it is a temporary memory

